In [1]:
import pandas as pd
import numpy as np
# from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

import matplotlib.pyplot as plt
import seaborn as sns

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler


# Explicación: Importante sobre la plantilla

Se tiene que tener en cuenta que este notebook no tiene ningún archivo asociado, ni se carga ningún DataFrame de prueba. Por lo tanto, es una plantilla que no debería ejecutarse como tal, es decir, de este apartado se debería poder identificar que código necesitan y usarlo en el notebook que entregarán en el parcial práctico.

Esta etapa del parcial es en la sala Waira. Deben utilizar los computadores de la sala y conectarse al usuario Examen. En los computadores está disponible Visual Studio Code con Python para editar y ejecutar los notebooks. 
Además, con el fin de que puedan resolver inquietudes de código, tendrán acceso a los siguientes sitios:

    https://scikit-learn.org/
    https://pandas.pydata.org/
    https://imbalanced-learn.org/stable/
    https://numpy.org/
    https://matplotlib.org/
    https://seaborn.pydata.org/
    https://pypi.org/

In [2]:
df = pd.read_csv('Dataset_practica_clasificacion.csv', encoding='utf-8')

In [ ]:

# =========================================================
# Análisis exploratorio y calidad de datos
# =========================================================


In [4]:
data = df.copy()

In [5]:
df.shape

(506, 14)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 506 entries, 0 to 505
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   log_id                  506 non-null    int64  
 1   Edad                    506 non-null    int64  
 2   Género                  502 non-null    str    
 3   Canal_preferido         502 non-null    str    
 4   Tipo_plan               503 non-null    str    
 5   Antigüedad_meses        502 non-null    float64
 6   Compras_ult_6m          503 non-null    float64
 7   Ticket_promedio         503 non-null    float64
 8   Reclamos_ult_12m        506 non-null    int64  
 9   Satisfacción            502 non-null    float64
 10  Descuento_promedio_pct  503 non-null    float64
 11  Nivel_uso_app           504 non-null    str    
 12  Fecha_ultima_compra     503 non-null    str    
 13  Renueva_membresia       495 non-null    str    
dtypes: float64(5), int64(3), str(6)
memory usage: 74.0 KB

In [7]:
df.describe(include='all')

,log_id,Edad,Género,Canal_preferido,Tipo_plan,Antigüedad_meses,Compras_ult_6m,Ticket_promedio,Reclamos_ult_12m,Satisfacción,Descuento_promedio_pct,Nivel_uso_app,Fecha_ultima_compra,Renueva_membresia
count,506.000000,506.000000,502,502,503,502.000000,503.000000,5.030000e+02,506.000000,502.000000,503.000000,504,503,495
unique,NaN,NaN,4,7,5,NaN,NaN,NaN,NaN,NaN,NaN,7,247,2
top,NaN,NaN,Masculino,Web,Estándar,NaN,NaN,NaN,NaN,NaN,NaN,Medio,2026-07-13,Sí
freq,NaN,NaN,257,214,215,NaN,NaN,NaN,NaN,NaN,NaN,199,7,258
mean,545784.017787,46.033597,NaN,NaN,NaN,54.533865,8.976143,1.066324e+05,0.709486,6.902590,17.181312,NaN,NaN,NaN
std,258937.058544,35.635126,NaN,NaN,NaN,36.297257,6.367828,8.100149e+04,1.126015,1.686643,10.632886,NaN,NaN,NaN
min,102874.000000,18.000000,NaN,NaN,NaN,1.000000,-2.000000,0.000000e+00,0.000000,-1.500000,-4.000000,NaN,NaN,NaN
25%,333904.500000,30.000000,NaN,NaN,NaN,21.000000,5.000000,6.325050e+04,0.000000,5.800000,10.400000,NaN,NaN,NaN
50%,533972.500000,43.000000,NaN,NaN,NaN,52.000000,9.000000,1.006300e+05,0.000000,6.900000,16.900000,NaN,NaN,NaN
75%,767738.750000,56.000000,NaN,NaN,NaN,89.000000,13.000000,1.430595e+05,1.000000,8.000000,23.500000,NaN,NaN,NaN


In [8]:
df.describe(include=['object', 'category'])

/var/folders/cp/p5kbj9_s7wv6hvd3xffs7nd40000gn/T/ipykernel_55929/3718493387.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include=['object', 'category'])


,Género,Canal_preferido,Tipo_plan,Nivel_uso_app,Fecha_ultima_compra,Renueva_membresia
count,502,502,503,504,503,495
unique,4,7,5,7,247,2
top,Masculino,Web,Estándar,Medio,2026-07-13,Sí
freq,257,214,215,199,7,258


In [9]:
df.columns

Index(['log_id', 'Edad', 'Género', 'Canal_preferido', 'Tipo_plan',
       'Antigüedad_meses', 'Compras_ult_6m', 'Ticket_promedio',
       'Reclamos_ult_12m', 'Satisfacción', 'Descuento_promedio_pct',
       'Nivel_uso_app', 'Fecha_ultima_compra', 'Renueva_membresia'],
      dtype='str')

In [10]:
df.isna().sum()

log_id                     0
Edad                       0
Género                     4
Canal_preferido            4
Tipo_plan                  3
Antigüedad_meses           4
Compras_ult_6m             3
Ticket_promedio            3
Reclamos_ult_12m           0
Satisfacción               4
Descuento_promedio_pct     3
Nivel_uso_app              2
Fecha_ultima_compra        3
Renueva_membresia         11
dtype: int64

# **Unicidad**

In [11]:
data.duplicated().sum()

np.int64(4)

In [13]:
data[data['log_id'].duplicated(keep=False)]


,log_id,Edad,Género,Canal_preferido,Tipo_plan,Antigüedad_meses,Compras_ult_6m,Ticket_promedio,Reclamos_ult_12m,Satisfacción,Descuento_promedio_pct,Nivel_uso_app,Fecha_ultima_compra,Renueva_membresia
14,285291,52,Masculino,App,Premium,10.0,13.0,159394.0,0,8.8,19.7,Medio,2026-04-30,Sí
120,220553,37,Masculino,Web,Premium,55.0,17.0,175490.0,0,5.3,19.2,Alto,2026-04-17,Sí
205,991014,29,Masculino,Tienda,Estándar,84.0,9.0,96703.0,2,7.3,8.5,Medio,2025-10-15,No
331,356256,41,Masculino,Web,Estándar,73.0,3.0,136719.0,4,4.4,15.4,Medio,2026-06-13,No
472,389759,23,Femenino,Tienda,Básico,15.0,5.0,38457.0,0,6.8,2.2,Bajo,2026-04-14,No
492,220553,50,Masculino,Web,Estándar,46.0,8.0,190490.0,2,5.0,18.4,Medio,2025-12-05,No
500,285291,52,Masculino,App,Premium,10.0,13.0,159394.0,0,8.8,19.7,Medio,2026-04-30,Sí
501,991014,29,Masculino,Tienda,Estándar,84.0,9.0,96703.0,2,7.3,8.5,Medio,2025-10-15,No
502,356256,41,Masculino,Web,Estándar,73.0,3.0,136719.0,4,4.4,15.4,Medio,2026-06-13,No
503,389759,23,Femenino,Tienda,Básico,15.0,5.0,38457.0,0,6.8,2.2,Bajo,2026-04-14,No


In [14]:
data.drop(columns='log_id').duplicated().sum()

np.int64(6)

In [15]:
data = data.drop_duplicates(subset=data.columns.drop('log_id'), keep='first').reset_index(drop=True)

In [16]:
data = data.drop_duplicates(subset=["log_id"],keep='last')

In [17]:
data.duplicated().sum()

np.int64(0)

# **Consistencia**

In [18]:
for c in data.select_dtypes('object'):
    print(data[c].value_counts(dropna=False), '\n')

Género
Masculino    251
Femenino     238
NaN            4
F              3
M              3
Name: count, dtype: int64 

Canal_preferido
Web       211
App       156
Tienda    122
NaN         4
 Web        2
web         2
tienda      1
app         1
Name: count, dtype: int64 

Tipo_plan
Estándar    212
Básico      170
Premium     111
NaN           3
estándar      2
Basico        1
Name: count, dtype: int64 

Nivel_uso_app
Medio      195
Bajo       168
Alto       127
MEDIO        2
NaN          2
VIP          2
 medio       2
ALTO         1
Name: count, dtype: int64 

Fecha_ultima_compra
2026-07-13    7
2026-08-26    6
2025-10-15    6
2026-08-30    6
2026-07-06    5
             ..
2026-08-22    1
2025-12-28    1
2025-11-20    1
2025-11-07    1
2026-08-04    1
Name: count, Length: 248, dtype: int64 

Renueva_membresia
Sí     254
No     234
NaN     11
Name: count, dtype: int64 



/var/folders/cp/p5kbj9_s7wv6hvd3xffs7nd40000gn/T/ipykernel_55929/2286750944.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in data.select_dtypes('object'):


In [19]:
data['Renueva_membresia'].value_counts()

Renueva_membresia
Sí    254
No    234
Name: count, dtype: int64

In [20]:
data['Renueva_membresia'].unique()


<ArrowStringArray>
['Sí', 'No', nan]
Length: 3, dtype: str

In [21]:
data['Género'].unique()


<ArrowStringArray>
['Femenino', 'Masculino', 'F', nan, 'M']
Length: 5, dtype: str

In [22]:
data.loc[data['Género'] == 'M', 'Género'] = 'Masculino'
data.loc[data['Género'] == 'F', 'Género'] = 'Femenino'

In [23]:
data['Canal_preferido'].unique()


<ArrowStringArray>
['Tienda', 'Web', 'App', 'tienda', nan, 'app', ' Web ', 'web']
Length: 8, dtype: str

In [24]:
data.loc[data['Canal_preferido'] == 'tienda', 'Canal_preferido'] = 'Tienda'
data.loc[data['Canal_preferido'] == 'app', 'Canal_preferido'] = 'App'
data.loc[data['Canal_preferido'] == ' Web ', 'Canal_preferido'] = 'Web'
data.loc[data['Canal_preferido'] == 'web', 'Canal_preferido'] = 'Web'

In [25]:
data['Tipo_plan'].unique()
	

<ArrowStringArray>
['Estándar', 'Básico', 'Premium', 'estándar', nan, 'Basico']
Length: 6, dtype: str

In [26]:
data.loc[data['Tipo_plan'] == 'estándar', 'Tipo_plan'] = 'Estándar'
data.loc[data['Tipo_plan'] == 'Basico', 'Tipo_plan'] = 'Básico'

In [27]:
data['Nivel_uso_app'].unique()

<ArrowStringArray>
['Alto', 'Bajo', 'Medio', 'MEDIO', nan, 'VIP', 'ALTO', ' medio ']
Length: 8, dtype: str

In [28]:
data["Nivel_uso_app"] = data["Nivel_uso_app"].replace(
    {
        "MEDIO" : "Medio",
        " medio " : "Medio",
        "ALTO" : "Alto",

    }
)

# **Validez**

In [29]:
data["Edad"] = np.where(data['Edad'] >= 100, data['Edad']/12 , data['Edad'])


In [30]:
data[data['Edad']<100]

,log_id,Edad,Género,Canal_preferido,Tipo_plan,Antigüedad_meses,Compras_ult_6m,Ticket_promedio,Reclamos_ult_12m,Satisfacción,Descuento_promedio_pct,Nivel_uso_app,Fecha_ultima_compra,Renueva_membresia
0,271137,44.0,Femenino,Tienda,Estándar,2.0,2.0,83078.0,0,10.0,11.9,Alto,2026-03-31,Sí
1,664930,48.0,Masculino,Web,Básico,90.0,2.0,31340.0,2,4.4,11.0,Bajo,2026-04-17,No
2,837775,67.0,Femenino,App,Básico,3.0,9.0,35035.0,0,6.7,10.7,Bajo,2026-07-13,Sí
3,498063,59.0,Femenino,Tienda,Estándar,26.0,12.0,112321.0,0,9.1,13.9,Bajo,2026-06-03,No
4,845394,70.0,Femenino,Web,Básico,12.0,9.0,122966.0,1,6.6,0.0,Medio,2026-07-06,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,397402,19.0,Femenino,App,Estándar,6.0,0.0,0.0,2,6.6,13.1,Bajo,2025-11-20,No
496,355783,63.0,Femenino,App,Estándar,86.0,0.0,0.0,0,8.8,4.3,Bajo,2026-01-17,Sí
497,828112,19.0,Masculino,Web,Básico,7.0,1.0,99873.0,1,5.9,4.2,Medio,2025-11-07,No
498,499935,46.0,Femenino,Tienda,Básico,91.0,4.0,41396.0,0,9.5,15.5,Alto,2026-08-04,Sí


In [31]:
data.loc[data['Compras_ult_6m'] == 0, 'Ticket_promedio'] = 0

In [54]:
Q1 = data["Ticket_promedio"].quantile(0.25)
Q3 = data["Ticket_promedio"].quantile(0.75)
IQR = Q3 - Q1
data = data[~((data["Ticket_promedio"] < (Q1 - 1.5 * IQR)) | (data["Ticket_promedio"] > (Q3 + 1.5 * IQR)))]

# **Completitud**

In [40]:
((data.isnull().sum()/data.shape[0])).sort_values(ascending=False)

Género                    0.008197
Canal_preferido           0.008197
Antigüedad_meses          0.008197
Satisfacción              0.008197
Tipo_plan                 0.006148
Compras_ult_6m            0.006148
Descuento_promedio_pct    0.006148
Fecha_ultima_compra       0.006148
Ticket_promedio           0.004098
Nivel_uso_app             0.004098
log_id                    0.000000
Edad                      0.000000
Reclamos_ult_12m          0.000000
Renueva_membresia         0.000000
dtype: float64

In [39]:
data = data.dropna(subset=["Renueva_membresia"])

### **Definir X y Y**

In [55]:
X = data.drop('Renueva_membresia', axis=1)
y = data['Renueva_membresia']

### **Train y test**

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

X_train_preparado = preprocessor.fit_transform(X_train)

X_train_df_prep = pd.DataFrame(
    X_train_preparado, 
    columns=preprocessor.get_feature_names_out()
)

X_test_preparado = preprocessor.transform(X_test)




------------------------------